# Train YOLO11-seg on Colab (M4)

Trains the RGB fire-segmentation baseline with `scripts/train_yolo.py` from the Vigil repo.

**Before running**
1. On the Mac: `python scripts/pack_yolo_for_colab.py` writes `data/flame2/colab/flame2_rgb_train_yn.zip` (about 1.75 GB).
2. Upload it to Google Drive as `MyDrive/vigil/flame2_rgb_train_yn.zip`.
3. Here: **Runtime → Change runtime type → T4 or L4 GPU**. An A100 is faster but uses compute units several times faster.

Then **Runtime → Run all**. Results and weights are saved to `MyDrive/vigil/runs/` after every epoch.

**Overnight:** on Colab Pro the run stops if this tab closes or the computer sleeps. Keep the tab open, the Mac plugged in, and stop it sleeping (in Terminal: `caffeinate -dis`, Ctrl+C to end). If the session drops anyway, set `RESUME = True` below and run all cells again: training carries on from the last finished epoch.

In [ ]:
RESUME = False  # True: carry on a run that was cut off
RUN_NAME = "rgb_baseline"  # results go to MyDrive/vigil/runs/<RUN_NAME>
BRANCH = "main"  # which version of the Vigil code to use
DRIVE_DIR = "/content/drive/MyDrive/vigil"
ZIP_NAME = "flame2_rgb_train_yn.zip"

In [ ]:
# Which GPU did this session get?
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# The training code, and the one package it needs (Colab already has PyTorch).
!rm -rf /content/Vigil && git clone --quiet --depth 1 --branch {BRANCH} https://github.com/WKS-DS/Vigil.git /content/Vigil
!pip install --quiet -r /content/Vigil/requirements-train.txt

In [ ]:
# Copy the dataset to this session's own disk: reading ~26,000 small files straight from Drive is very slow.
!mkdir -p /content/flame2_rgb && cp "{DRIVE_DIR}/{ZIP_NAME}" /content/ && unzip -q -o "/content/{ZIP_NAME}" -d /content/flame2_rgb && rm "/content/{ZIP_NAME}"
!ls /content/flame2_rgb && wc -l /content/flame2_rgb/train_yn.txt /content/flame2_rgb/val.txt

In [ ]:
resume_flag = "--resume" if RESUME else ""
!python /content/Vigil/scripts/train_yolo.py --data-dir /content/flame2_rgb --out "{DRIVE_DIR}/runs" --name {RUN_NAME} {resume_flag}

## Results

`MyDrive/vigil/runs/<RUN_NAME>/` holds `weights/best.pt`, `results.csv` (one row per epoch) and plots. Copy `best.pt` to the Mac's `data/weights/` for the fire-frame evaluation (next PR).

Ultralytics' validation score here includes the no-fire frames, which have near-copies in training, so it reads high. Treat it as a progress bar, not the result.